# Quickstart

Use `onceover.profile(df)` to inspect an orders table for candidate keys,
relationships between columns, and duplicate rows.

````{admonition} Minimal example
:class: tip

```python
import polars as pl
import onceover

report = onceover.profile(df)      # budget="standard" by default
print(report.overview["headline"]) # one-glance summary
report.findings                    # ranked statements with evidence
report.columns                     # per-column cards
report.coverage                    # what was examined, skipped, capped
```
````

The fixture contains a primary key, a region lookup, a nearly-unique
external reference with two repeated values, and a derived column. This
known structure lets us check the report against the input.

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 2000

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]   # two duplicated values
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})
df.head(4)

## Profile it

One call runs the detectors enabled by the default budget.
[](large_tables.ipynb) describes runtime and coverage on larger tables.

In [ ]:
report = onceover.profile(df)
print(report.overview["headline"])

The headline reports a key candidate, five functional dependencies,
predictable columns, and conditional rules for this fixture.

## Inspect findings

`report.findings` is sorted by its priority score, `interestingness`:


In [ ]:
for f in report.findings[:8]:
    print(f"[{f['interestingness']:.2f}] {f['claim'][:95]}")

The report verifies mappings between `quantity` and `double_quantity`,
checks their affine formula, and finds the region lookup. It also reports
that `order_id` is unique while `external_ref` has two repeated values.

The following count shows the categories present in this report:


In [ ]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

To inspect findings about one column, filter by `subject.columns`. For example, these findings refer to `external_ref`:

In [ ]:
for f in report.findings:
    if "external_ref" in f["subject"]["columns"]:
        print(f"[{f['category']}] {f['claim']}")

`external_ref` looks like an identifier, but it contains two repeated
values. Joining on it as though it were unique can produce extra rows.
Inspect the uniqueness finding before choosing it as a key.

## Inspect a complete finding


In [ ]:
f = report.findings[0]
print(json.dumps(f, indent=1)[:1600])

The result names its columns in `subject`. `specification` describes the
comparison, mapping, or formula, and `evidence` contains the evaluated row
counts and metrics. The [findings chapter](findings.ipynb) explains these
fields and shows how to filter on them.


In [ ]:
identity = next(f for f in report.findings if "within tolerance" in f["claim"])
print(identity["claim"])
print("category:", identity["category"], "| type:", identity.get("type"))
print("form:", identity["specification"].get("form"))
ev = identity["evidence"][0]
print("basis:", ev["basis"], "| sampling:", ev["scope"]["sampling"])
print("validation:", ev["validation"]["method"])
print("metrics:", ev["metrics"])

The identity check fitted `y = a*x + b` on a sample, then checked the
residuals on all 2,000 rows. Every row matched within the stated tolerance.

## Inspect column summaries

`report.columns` contains counts, statistics, and suggested roles, keyed
by column name. For example:


In [ ]:
print(json.dumps(report.columns["region_id"], indent=1)[:900])
print(json.dumps(report.columns["double_quantity"], indent=1)[:700])

`region_id` has three distinct numeric values and receives a categorical
role with a `treat_as_categorical` suggestion. `double_quantity` includes
exact distribution summaries. These counts use every input row; see
[Column summaries](columns.ipynb) for the available fields.

## Check which passes ran

`report.coverage` lists completed and skipped passes, along with their
sample sizes and limits:


In [ ]:
for name, info in report.coverage.items():
    print(f"{name:28s} {info['status']:14s} {info['detail'][:72]}")

`row_duplicates` can rule out duplicate rows without comparing whole
rows because `order_id` has already been found unique and null-free.
`functional_dependencies` reports how many rows it checked. This fixture
fits within its verification cap; larger inputs may be sampled.

## Save the report

Use `save()` to write a report. JSON output is repeatable for the same
package version, data, and settings when no deadline is set.


In [ ]:
import os
import tempfile

path = report.save(tempfile.mkdtemp() + "/profile.json")
print("saved", os.path.getsize(path), "bytes to", path)
print(report.to_markdown()[:400])

`save()` writes JSON by default; `report.data` returns the parsed report.
`to_markdown()` produces a text summary suitable for sharing in an issue
or pull request.

## Further examples

- [Reading findings](findings.ipynb): fields, scores, and filtering.
- [Column summaries](columns.ipynb): roles and distribution statistics.
- [Coverage](coverage.ipynb): budgets, sampling, and detector selection.
- [Analysis notebook](https://github.com/fcela/onceover/blob/main/notebooks/onceover_analysis.ipynb):
  a larger fixture with charts and checks against planted relationships.
